# Course 8: Streamlined Data Ingestion with pandas

This notebook records core concepts, syntax patterns, edge-case handling, and practical workflows from the **Streamlined Data Ingestion with pandas** course on DataCamp (instructed by Amany Mahfouz).

---
## Chapter 1: Importing Data from Flat Files

### 1.1 DataFrames & Flat File Formats
* **pandas `DataFrame`**: Two-dimensional tabular data structure with labeled columns and row labels (`Index`).
* **Flat Files**: Plain-text files with no visual formatting where each line represents one row and fields are separated by a delimiter:
  * **CSV (Comma-Separated Values)**: Fields separated by commas (`,`).
  * **TSV / Custom Delimited Files**: Fields separated by tabs (`\t`), pipes (`|`), or semicolons (`;`).
* **`pd.read_csv()`**: Universal `pandas` reader for flat files. Use `sep="\t"` (or any custom delimiter string) when reading non-comma-separated files.

### 1.2 Modifying Flat File Imports (Subsetting Columns, Rows & Headers)
* **Selecting Columns (`usecols`)**: Restricts loaded columns to reduce memory and speed up imports. Accepts:
  * A list of column names (`['STATEFIPS', 'STATE', 'zipcode', 'agi_stub', 'N1']`).
  * A list of zero-indexed column positions (`[0, 1, 2, 3, 4]`).
  * A callable filter function (e.g., `lambda col: col.startswith('N')`).
* **Limiting & Chunking Rows (`nrows` & `skiprows`)**:
  * `nrows=1000`: Reads only the first 1,000 data rows.
  * `skiprows=100`: Skips the first 100 rows (also accepts a list of specific row indices or a filter function).
  * **Chunking Caveat**: When using `skiprows` on a file with a header row, row `0` (the header) is skipped too! Pass `header=None` and supply column names via `names=col_names` (where `len(col_names)` must match the exact number of columns in the file).

### 1.3 Handling Data Types, Custom Missing Values & Corrupt Lines
* **Specifying Data Types (`dtype`)**: Prevents `pandas` from misinferring codes (such as leading-zero ZIP codes) as integers by passing a dictionary: `dtype={"zipcode": str}`.
* **Custom Missing Values (`na_values`)**: Flags sentinel values (like `0` or `"n/a"`) as `NaN` during load. Accepts a scalar, list, or column-mapped dictionary (`na_values={"zipcode": 0}`).
* **Handling Unparseable Records (`on_bad_lines` / `error_bad_lines`)**:
  * Lines with mismatched field counts raise `pandas.errors.ParserError`.
  * In **modern `pandas` (1.3+ / 2.x)**: Use `on_bad_lines='warn'` (skips bad rows and prints a warning) or `on_bad_lines='skip'`.
  * In **legacy `pandas` (<1.3)**: Shown in course slides as `error_bad_lines=False, warn_bad_lines=True`.

In [1]:
# 1.1 & 1.2 Loading CSV/TSV files and subsetting columns & rows
import pandas as pd

# Load standard CSV and tab-separated TSV
tax_data = pd.read_csv("datasets/us_tax_data_2016.csv")
tax_data_tsv = pd.read_csv("datasets/us_tax_data_2016.tsv", sep="\t")
print("CSV equals TSV:", tax_data.equals(tax_data_tsv))
print("Shape:", tax_data.shape)

# Limiting columns by name vs by column position
col_names = ['STATEFIPS', 'STATE', 'zipcode', 'agi_stub', 'N1']
col_nums = [0, 1, 2, 3, 4]

tax_data_v1 = pd.read_csv("datasets/us_tax_data_2016.csv", usecols=col_names)
tax_data_v2 = pd.read_csv("datasets/us_tax_data_2016.csv", usecols=col_nums)
print("Column subsetting equivalent:", tax_data_v1.equals(tax_data_v2))

# Chunking rows with nrows, skiprows, header=None, and names
all_cols = list(tax_data)
tax_data_chunk = pd.read_csv(
    "datasets/us_tax_data_2016.csv",
    nrows=10,
    skiprows=5,
    header=None,
    names=all_cols
)
print("\nChunked rows (rows 5-14) with restored column names:")
print(tax_data_chunk[['STATEFIPS', 'STATE', 'zipcode', 'agi_stub']].head(3))

CSV equals TSV: True
Shape: (96, 10)
Column subsetting equivalent: True

Chunked rows (rows 5-14) with restored column names:
   STATEFIPS STATE  zipcode  agi_stub
0          1    AL        0         5
1          1    AL        0         6
2          1    AL    35565         1


In [2]:
# 1.3 Specifying dtypes, custom na_values, and skipping corrupt records
# 1. Cast zipcode to string and treat 0 as missing (NaN)
tax_clean = pd.read_csv(
    "datasets/us_tax_data_2016.csv",
    dtype={"zipcode": str},
    na_values={"zipcode": "0"}
)
print("zipcode dtype:", tax_clean["zipcode"].dtype)
print("Missing zipcode rows count:", tax_clean["zipcode"].isna().sum())

# 2. Handling corrupt lines with mismatched field counts
# (Modern pandas >= 1.3 uses on_bad_lines='warn'; legacy pandas used error_bad_lines=False, warn_bad_lines=True)
tax_corrupt_handled = pd.read_csv(
    "datasets/us_tax_data_2016_corrupt.csv",
    on_bad_lines="warn"
)
print("Loaded clean rows after skipping corrupt line:", len(tax_corrupt_handled))

zipcode dtype: object
Missing zipcode rows count: 24
Loaded clean rows after skipping corrupt line: 23


### Chapter 1: Key Takeaways & Actionable Step-by-Step Process

#### Key Takeaways
* `pd.read_csv()` handles all delimited flat files via the `sep` parameter (`sep="\t"` for TSVs).
* Use `usecols` and `nrows` to trim wide or massive files at ingestion time rather than filtering in memory afterward.
* When paginating through a file with `skiprows` and `nrows`, always pass `header=None` and `names=col_names` so the first data row is not mistaken for column headers.
* Enforce schema integrity on load with `dtype` (e.g., preserving leading zeros in IDs/ZIP codes) and `na_values` (converting sentinel numbers like `0` to `NaN`).

#### Step-by-Step Process to Act On
1. **Inspect Schema First**: Load a small sample (`pd.read_csv(path, nrows=100)`) to verify delimiter (`sep`), headers, and inferred `.dtypes`.
2. **Prune Unneeded Data**: Pass `usecols=[...]` to load only relevant columns.
3. **Enforce Types & Nulls**: Pass `dtype={...}` for string identifiers and `na_values={...}` for placeholder values.
4. **Guard Against Malformed Rows**: Add `on_bad_lines="warn"` when ingesting unvalidated user or third-party flat files.

---
### Chapter 1: Glossary & Official Documentation

| Item | Category | Explanation | Arguments Accepted | Official Documentation |
| :--- | :--- | :--- | :--- | :--- |
| `pd.read_csv()` | `pandas` I/O Function | Reads a comma-separated values (or custom-delimited) flat file into a `DataFrame`. | `filepath_or_buffer`, `sep=','`, `header='infer'`, `names`, `usecols`, `dtype`, `nrows`, `skiprows`, `na_values`, `on_bad_lines='error'` | [Pandas `read_csv` Docs](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) |
| `usecols` | `read_csv` Parameter | Restricts imported columns by name, positional index, or callable filter. | `list` of `str`/`int`, or `callable` | [Pandas `read_csv` (`usecols`)](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) |
| `na_values` | `read_csv` Parameter | Additional strings or numbers to recognize as `NA`/`NaN` during parsing. | `scalar`, `str`, `list-like`, or `dict` | [Pandas `read_csv` (`na_values`)](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) |
| `on_bad_lines` | `read_csv` Parameter | Specifies action when encountering a line with too many fields (`'error'`, `'warn'`, `'skip'`). | `'error'`, `'warn'`, `'skip'`, or `callable` | [Pandas `read_csv` (`on_bad_lines`)](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) |

---
## Chapter 2: Importing Data from Excel Files

### 2.1 Loading Spreadsheets & Selecting Rows/Columns (`pd.read_excel()`)
* **Excel Workbooks**: Tabular files (`.xlsx`, `.xls`) that support cell formatting, formulas, metadata headers, and multiple sheets.
* **`pd.read_excel()`**: Shares `nrows`, `skiprows`, and `usecols` with `read_csv()`, plus Excel-specific column letter ranges:
  * Example: `pd.read_excel("fcc_survey_with_headers.xlsx", skiprows=2, usecols="W:AB, AR")` skips 2 title rows and loads Excel columns `W` through `AB` plus `AR`.

### 2.2 Getting Data from Multiple Worksheets (`sheet_name`)
* By default, `pd.read_excel()` loads only the first worksheet (`sheet_name=0`).
* **Selecting Specific Sheets**: Pass a sheet name string (`sheet_name='2017'`), a zero-based integer index (`sheet_name=1`), or a list of names/indices (`sheet_name=['2016', 1]`).
* **Loading All Sheets (`sheet_name=None`)**: Returns an `OrderedDict` (or `dict`) mapping `{sheet_name: DataFrame}`. Iterate over `.items()`, tag each frame with its sheet name (`frame["Year"] = sheet_name`), and combine with `pd.concat()`.

### 2.3 Modifying Imports: Boolean Data (`dtype`, `true_values`, `false_values`)
* **Default Behavior Pitfall**: `pandas` loads `1`/`0` or `TRUE`/`FALSE` columns with missing values as `float64`.
* **Casting to `bool` (`dtype={"col": bool}`)**:
  * **Critical Caveat**: Python `bool` columns **cannot hold `NaN` values**! Any `NaN` or unrecognized string value in a column cast to `bool` is automatically coerced to `True`.
  * Use `true_values=["Yes"]` and `false_values=["No"]` alongside `dtype={...}` so custom text indicators map properly instead of defaulting to `True`.
  * **Design Check**: If missing values (`NaN`) carry analytical meaning, keep the column as `float64` or nullable `boolean` rather than NumPy `bool`.

### 2.4 Modifying Imports: Parsing Dates (`parse_dates` & `pd.to_datetime()`)
* **Standard Datetimes (`parse_dates`)**: Use `parse_dates` (not `dtype`!) to parse standard date strings into `datetime64[ns]`:
  * **List of columns**: `parse_dates=["Part1StartTime", "Part1EndTime"]`
  * **Nested list to combine split date & time columns**: `parse_dates=[["Part2StartDate", "Part2StartTime"]]`
  * **Dictionary to combine & rename**: `parse_dates={"Part2Start": ["Part2StartDate", "Part2StartTime"]}` (note: in `pandas` 2.x, combining columns via `parse_dates` is deprecated in favor of `pd.to_datetime(df['Date'] + ' ' + df['Time'])`).
* **Non-Standard Date Formats (`pd.to_datetime()`)**: When strings use custom layouts (e.g., `"03292016 21:27:25"`), load as string and parse with explicit `strftime` codes:
  * `%Y`: 4-digit year (`2016`) | `%m`: zero-padded month (`03`) | `%d`: zero-padded day (`29`)
  * `%H`: 24-hour clock (`21`) | `%M`: zero-padded minute (`27`) | `%S`: zero-padded second (`25`)

In [3]:
# 2.1 & 2.2 Loading Excel spreadsheets, Excel column ranges, and multiple worksheets
import pandas as pd

# 1. Load select columns using Excel letter notation (W:AB, AR) and skip metadata header rows
survey_subset = pd.read_excel(
    "datasets/fcc_survey_with_headers.xlsx",
    skiprows=2,
    usecols="W:AB, AR"
)
print("Subset from Excel letter range W:AB, AR:")
print(survey_subset.head(3))

# 2. Load specific sheet by index vs by name
sheet2_by_idx = pd.read_excel("datasets/fcc_survey.xlsx", sheet_name=1)
sheet2_by_name = pd.read_excel("datasets/fcc_survey.xlsx", sheet_name="2017")
print("\nSheet 1 equals Sheet '2017':", sheet2_by_idx.equals(sheet2_by_name))

# 3. Load ALL sheets with sheet_name=None and combine with pd.concat()
survey_responses = pd.read_excel("datasets/fcc_survey.xlsx", sheet_name=None)
all_responses = pd.DataFrame()
for sheet_name, frame in survey_responses.items():
    frame["Year"] = sheet_name
    all_responses = pd.concat([all_responses, frame], ignore_index=True)

print("Combined sheets shape:", all_responses.shape, "| Years:", all_responses["Year"].unique())

Subset from Excel letter range W:AB, AR:
   CommuteTime            CountryCitizen               CountryLive                    EmploymentField  EmploymentFieldOther    EmploymentStatus  Income
0           35  United States of America  United States of America  office and administrative support                   NaN  Employed for wages   32000
1           90  United States of America  United States of America                  food and beverage                   NaN  Employed for wages   15000
2           45  United States of America  United States of America                            finance                   NaN  Employed for wages   48000

Sheet 1 equals Sheet '2017': True
Combined sheets shape: (20, 10) | Years: ['2016' '2017']


In [4]:
# 2.3 & 2.4 Boolean casting with custom true/false values & Date/Time parsing
# 1. Custom Boolean values:
#    - Non-null binary columns can be cast directly with `dtype=bool` + `true_values` / `false_values`
#    - Columns with missing values (NA) should be loaded with `true_values`/`false_values` and cast via `.astype("boolean")`
bool_data = pd.read_excel(
    "datasets/fcc_survey_booleans.xlsx",
    dtype={
        "AttendedBootcamp": bool,
        "AttendedBootCampYesNo": bool,
        "AttendedBootcampTF": bool,
    },
    true_values=["Yes"],
    false_values=["No"]
)
for col in ["BootcampLoan", "LoanYesNo", "LoanTF"]:
    bool_data[col] = bool_data[col].astype("boolean")

print("Boolean True counts (nullable 'boolean' preserves <NA> cleanly):")
print(bool_data.drop(columns=["ID.x"]).sum())
print("\nMissing (<NA>) counts per column:")
print(bool_data.drop(columns=["ID.x"]).isna().sum())

# 2. Parsing standard dates with parse_dates and non-standard dates with pd.to_datetime()
survey_dates = pd.read_excel(
    "datasets/fcc_survey.xlsx",
    parse_dates=["Part1StartTime", "Part1EndTime"]
)

# Combine split date + time columns cleanly
survey_dates["Part2Start"] = pd.to_datetime(
    survey_dates["Part2StartDate"].astype(str) + " " + survey_dates["Part2StartTime"].astype(str)
)

# Parse non-standard format "%m%d%Y %H:%M:%S" ("03292016 21:27:25")
survey_dates["Part2EndTime"] = pd.to_datetime(
    survey_dates["Part2EndTime"],
    format="%m%d%Y %H:%M:%S"
)

print("\nParsed Datetime Dtypes:")
print(survey_dates[["Part1StartTime", "Part1EndTime", "Part2Start", "Part2EndTime"]].dtypes)
print(survey_dates[["Part2Start", "Part2EndTime"]].head(3))

Boolean True counts (nullable 'boolean' preserves <NA> cleanly):
AttendedBootcamp         3
AttendedBootCampYesNo    3
AttendedBootcampTF       3
BootcampLoan             1
LoanYesNo                1
LoanTF                   1
dtype: int64

Missing (<NA>) counts per column:
AttendedBootcamp         0
AttendedBootCampYesNo    0
AttendedBootcampTF       0
BootcampLoan             7
LoanYesNo                7
LoanTF                   7
dtype: int64

Parsed Datetime Dtypes:
Part1StartTime    datetime64[ns]
Part1EndTime      datetime64[ns]
Part2Start        datetime64[ns]
Part2EndTime      datetime64[ns]
dtype: object
           Part2Start        Part2EndTime
0 2016-03-29 21:24:57 2016-03-29 21:27:25
1 2016-03-29 21:27:14 2016-03-29 21:29:10
2 2016-03-29 21:27:13 2016-03-29 21:28:21


### Chapter 2: Key Takeaways & Actionable Step-by-Step Process

#### Key Takeaways
* `pd.read_excel()` supports Excel-style column ranges (`usecols="W:AB, AR"`) and multi-sheet ingestion via `sheet_name=None`.
* Casting columns to NumPy `bool` silently converts `NaN` and unrecognized strings into `True`—always pair `dtype={...: bool}` with `true_values` and `false_values`, and avoid `bool` if missing values must remain distinct.
* Use `parse_dates` during `read_excel()` for ISO-standard timestamps, and `pd.to_datetime(..., format=...)` post-load for non-standard date strings.

#### Step-by-Step Process to Act On
1. **Audit Workbook Structure**: Check sheet names (`pd.ExcelFile(path).sheet_names`) and top rows to identify metadata headers (`skiprows`) and target columns (`usecols`).
2. **Consolidate Multi-Sheet Workbooks**: Load with `sheet_name=None`, tag each DataFrame with its sheet identifier (`frame["Sheet"] = name`), and combine via `pd.concat(..., ignore_index=True)`.
3. **Validate Binary Columns Before Casting**: Check `.isna().sum()` before forcing `dtype=bool`; if `NaN`s exist, keep `float64` or map explicitly.
4. **Standardize Timestamps**: Parse standard date columns with `parse_dates=[...]` and custom formats with `pd.to_datetime(col, format="%m%d%Y %H:%M:%S")`.

---
### Chapter 2: Glossary & Official Documentation

| Item | Category | Explanation | Arguments Accepted | Official Documentation |
| :--- | :--- | :--- | :--- | :--- |
| `pd.read_excel()` | `pandas` I/O Function | Reads an Excel `.xlsx`/`.xls` spreadsheet into a `DataFrame` (or dict of DataFrames). | `io`, `sheet_name=0`, `usecols`, `skiprows`, `nrows`, `dtype`, `true_values`, `false_values`, `parse_dates` | [Pandas `read_excel` Docs](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) |
| `sheet_name` | `read_excel` Parameter | Controls which worksheet(s) to load (`0` for first, `str` name, `list`, or `None` for all). | `str`, `int`, `list`, or `None` | [Pandas `read_excel` (`sheet_name`)](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) |
| `true_values` / `false_values` | `read_excel` Parameters | Lists of custom string values to recognize as `True` and `False` in `bool` columns. | `list` of `str` | [Pandas `read_excel` Docs](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) |
| `pd.to_datetime()` | `pandas` Function | Converts scalar, array-like, `Series`, or `DataFrame` to `datetime64` objects. | `arg`, `format=None`, `errors='raise'`, `dayfirst=False` | [Pandas `to_datetime` Docs](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) |
| `pd.concat()` | `pandas` Function | Concatenates `pandas` objects along a particular axis (`axis=0` stacks rows). | `objs`, `axis=0`, `join='outer'`, `ignore_index=False` | [Pandas `concat` Docs](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) |

---
## Chapter 3: Importing Data from Databases

### 3.1 Relational Databases & Connecting with `SQLAlchemy`
* **Relational Databases**: Organize entity data into linked tables (rows = records, columns = typed attributes) connected via primary/foreign keys and queried via **SQL (Structured Query Language)**.
* **Two-Step Ingestion Workflow**:
  1. **Create Database Engine**: `from sqlalchemy import create_engine; engine = create_engine("sqlite:///data.db")` (SQLite URLs use three slashes `sqlite:///` for relative file paths).
  2. **Query Database**: `pd.read_sql(query_or_table, engine)` executes a SQL query (or loads an entire table by name) directly into a `DataFrame`.

### 3.2 Refining Imports with SQL (`SELECT`, `WHERE`, `AND`, `OR`)
* Pushing filtering down to the database engine avoids transferring unnecessary rows/columns into Python memory:
  * **Column Selection**: `SELECT date, tavg FROM weather;`
  * **Row Filtering (`WHERE`)**: Supports numeric comparisons (`=`, `>`, `>=`, `<`, `<=`, `<>`) and case-sensitive text matching (`WHERE borough = 'BROOKLYN'`).
  * **Compound Conditions**: Combine conditions with `AND` (all conditions must be true) and `OR` (at least one condition must be true).

### 3.3 Aggregations & Grouping in SQL (`DISTINCT`, `COUNT`, `AVG`, `GROUP BY`)
* **`SELECT DISTINCT`**: Deduplicates returned values (`SELECT DISTINCT incident_address, borough FROM hpd311calls;`).
* **SQL Aggregate Functions**: `SUM(col)`, `AVG(col)`, `MAX(col)`, `MIN(col)`, `COUNT(*)`, and `COUNT(DISTINCT col)`.
* **`GROUP BY`**: Partitions rows into categories before applying aggregate functions (always include the grouping column in the `SELECT` clause).

### 3.4 Loading Multiple Tables with `JOIN`
* Combine normalized tables on matching key columns (ensure join keys share the same data type and qualify shared column names using `table.column` dot notation).
* **SQL Clause Execution / Syntax Order**:
  `SELECT` $\rightarrow$ `FROM` $\rightarrow$ `JOIN ... ON ...` $\rightarrow$ `WHERE` $\rightarrow$ `GROUP BY`

In [5]:
# 3.1 & 3.2 Connecting to SQLite via SQLAlchemy and filtering with WHERE, AND, OR
import sqlite3
import pandas as pd

# Step 1: Create SQLAlchemy database engine (with sqlite3 fallback if sqlalchemy is not installed)
try:
    from sqlalchemy import create_engine
    engine = create_engine("sqlite:///datasets/data.db")
except ImportError:
    engine = sqlite3.connect("datasets/data.db")

# Step 2: Load entire table by name or via SQL SELECT query
weather_df = pd.read_sql("SELECT * FROM weather;", engine)
print("Weather table preview:")
print(weather_df[['station', 'date', 'tmax', 'tmin']].head(3))

# Filtering with WHERE, AND, OR
query_filtered = """
SELECT unique_key, created_date, borough, complaint_type
FROM hpd311calls
WHERE (borough = 'BRONX' OR borough = 'BROOKLYN')
  AND complaint_type = 'PLUMBING';
"""
filtered_calls = pd.read_sql(query_filtered, engine)
print("\nFiltered Plumbing Calls in Bronx or Brooklyn:")
print(filtered_calls)

Weather table preview:
       station        date  tmax  tmin
0  USW00094728  01/01/2018    19     7
1  USW00094728  01/02/2018    26    13
2  USW00094728  01/03/2018    30    16

Filtered Plumbing Calls in Bronx or Brooklyn:
   unique_key created_date borough complaint_type
0    38066653   01/01/2018   BRONX       PLUMBING


In [6]:
# 3.3 & 3.4 SQL Aggregations (GROUP BY, COUNT) and Multi-Table JOINs
# 1. Aggregating call counts by borough with GROUP BY
agg_query = """
SELECT borough, COUNT(*) AS plumbing_count
FROM hpd311calls
WHERE complaint_type = 'PLUMBING'
GROUP BY borough;
"""
plumbing_counts = pd.read_sql(agg_query, engine)
print("Plumbing call counts by borough:")
print(plumbing_counts)

# 2. Joining hpd311calls with boro_census and aggregating
join_agg_query = """
SELECT hpd311calls.borough,
       COUNT(*) AS total_calls,
       boro_census.total_population,
       boro_census.housing_units
FROM hpd311calls
JOIN boro_census
  ON hpd311calls.borough = boro_census.borough
GROUP BY hpd311calls.borough;
"""
calls_with_census = pd.read_sql(join_agg_query, engine)
print("\nJoined Call Counts with Borough Census Data:")
print(calls_with_census)

Plumbing call counts by borough:
         borough  plumbing_count
0          BRONX               1
1      MANHATTAN               1
2         QUEENS               1
3  STATEN ISLAND               1

Joined Call Counts with Borough Census Data:
         borough  total_calls  total_population  housing_units
0          BRONX            2           1455846         524488
1       BROOKLYN            2           2635121        1028383
2      MANHATTAN            2           1653877         872645
3         QUEENS            1           2339280         850422
4  STATEN ISLAND            1            475948         179179


### Chapter 3: Key Takeaways & Actionable Step-by-Step Process

#### Key Takeaways
* `sqlalchemy.create_engine()` abstracts database connection strings (`sqlite:///path/to/file.db`), enabling `pd.read_sql()` to work seamlessly across SQLite, PostgreSQL, MySQL, Oracle, and SQL Server.
* Push filtering (`WHERE`), deduplication (`SELECT DISTINCT`), aggregations (`COUNT`, `AVG`, `GROUP BY`), and relational lookups (`JOIN`) into SQL so Python only ingests compact, analysis-ready results.
* Always qualify column names (`table.column`) in `JOIN` queries and verify that join keys share identical data types.

#### Step-by-Step Process to Act On
1. **Initialize Engine**: Instantiate `engine = create_engine("sqlite:///filename.db")` (or dialect URI).
2. **Draft & Validate SQL Query**: Write explicit `SELECT col1, col2 FROM ... WHERE ...` statements rather than `SELECT *` on large tables.
3. **Aggregate & Join Server-Side**: Use `JOIN ... ON ...` and `GROUP BY` in the query string to combine lookup tables and summarize metrics before loading into pandas.
4. **Execute into DataFrame**: Load results with `df = pd.read_sql(query, engine)`.

---
### Chapter 3: Glossary & Official Documentation

| Item | Category | Explanation | Arguments Accepted | Official Documentation |
| :--- | :--- | :--- | :--- | :--- |
| `create_engine()` | `sqlalchemy` Function | Creates a `SQLAlchemy` `Engine` instance to manage database connections. | `url` (`str`, e.g., `'sqlite:///data.db'`), `**kwargs` | [SQLAlchemy `create_engine` Docs](https://docs.sqlalchemy.org/en/20/core/engines.html#sqlalchemy.create_engine) |
| `pd.read_sql()` | `pandas` I/O Function | Reads a SQL query or database table into a `DataFrame`. | `sql` (`str` or SQLAlchemy Selectable), `con` (Engine/Connection), `index_col`, `parse_dates` | [Pandas `read_sql` Docs](https://pandas.pydata.org/docs/reference/api/pandas.read_sql.html) |
| `SELECT DISTINCT` | SQL Clause | Returns only unique combinations of values across the specified columns. | Column names (`SELECT DISTINCT col1, col2 ...`) | [SQLite `SELECT` Docs](https://www.sqlite.org/lang_select.html) |
| `GROUP BY` | SQL Clause | Groups rows sharing the same values into summary rows for aggregate functions (`COUNT`, `AVG`, `SUM`, `MIN`, `MAX`). | Grouping column expression(s) | [SQLite `GROUP BY` Docs](https://www.sqlite.org/lang_select.html#resultset) |
| `JOIN ... ON` | SQL Clause | Combines rows from two or more tables based on a related key column (`INNER JOIN` by default). | Right table name + `ON left.key = right.key` | [SQLite `JOIN` Docs](https://www.sqlite.org/lang_select.html#fromclause) |

---
## Chapter 4: Importing JSON Data and Working with APIs

### 4.1 Introduction to JSON & Data Orientations (`pd.read_json()`)
* **JSON (JavaScript Object Notation)**: Semi-structured, non-tabular web data format composed of collections of objects (`{key: value}`) and arrays (`[...]`), where records need not share identical attributes.
* **`pd.read_json()`**: Accepts a file path or a JSON string (supports `dtype` and `orient`).
* **Common JSON Orientations (`orient`)**:
  * `'records'` (most common): List of row dictionaries `[{"col1": val1, "col2": val2}, ...]`.
  * `'columns'`: Space-efficient dictionary of column dictionaries `{"col1": {"0": val1, "1": val2}}`.
  * `'split'`: Dictionary separating `"columns"`, `"index"`, and `"data"` arrays—must be explicitly flagged via `pd.read_json("nyc_death_causes.json", orient="split")`.

### 4.2 Introduction to APIs & `requests`
* **API (Application Programming Interface)**: Defines how programs request data from external services over HTTP without direct database access.
* **`requests.get(url, params=..., headers=...)`**:
  * `params`: Dictionary of query string parameters (e.g., `{"term": "bookstore", "location": "San Francisco"}`).
  * `headers`: Dictionary of HTTP headers, typically used for Bearer token authentication (`{"Authorization": f"Bearer {api_key}"}`).
  * **`response.json()`**: Parses the HTTP response body into a Python `dict` / `list`.
  * **Crucial Pitfall**: Do **not** pass `response.json()` into `pd.read_json()` (which expects a JSON *string* or file path and will raise a `TypeError`/`ValueError`); pass the parsed dictionary to `pd.DataFrame()` or `pd.json_normalize()`.

### 4.3 Flattening Nested JSONs (`pd.json_normalize()`)
* API responses frequently nest objects inside objects (e.g., `coordinates: {latitude, longitude}`) or lists of objects inside records (e.g., `categories: [{alias, title}]`).
* **`pd.json_normalize()`** (available directly as `pd.json_normalize()` in modern `pandas`, formerly `from pandas.io.json import json_normalize`):
  * `sep="_"`: Sets separator for flattened nested keys (default is `"."`, e.g., `coordinates.latitude` $\rightarrow$ `coordinates_latitude`).
  * `record_path="categories"`: Unpacks a nested list of objects so each nested item becomes its own row.
  * `meta=[...]`: Carries parent-level attributes (including nested paths like `["coordinates", "latitude"]`) onto each unpacked child row.
  * `meta_prefix="biz_"`: Prefixes metadata columns to prevent naming collisions (e.g., `alias` vs `biz_alias`).

### 4.4 Combining Multiple Datasets (`pd.concat()` & `df.merge()`)
* **Concatenating Rows (`pd.concat([df1, df2], ignore_index=True)`)**: Stacks paginated API results (e.g., `offset=0` and `offset=20`) vertically and resets the index.
* **Merging Columns (`df1.merge(df2, ...)`)**: Performs SQL-style joins across datasets sharing key columns:
  * Use `on="col"` when key names match, or `left_on="created_date", right_on="date"` when key names differ.
  * Join keys must have identical data types; by default `merge()` performs an inner join returning matching records.

In [7]:
# 4.1 Reading Split-Oriented JSON & 4.2 Parsing API Responses
import json
import pandas as pd

# 1. Reading split-oriented JSON (nyc_death_causes.json)
death_causes = pd.read_json("datasets/nyc_death_causes.json", orient="split")
print("Split-oriented JSON loaded into DataFrame:")
print(death_causes.head())

# 2. Simulating requests.get(...).json() with Yelp Business Search API payload
# In live code:
# response = requests.get(api_url, params={"term": "bookstore", "location": "San Francisco"}, headers={"Authorization": f"Bearer {api_key}"})
# data = response.json()
with open("datasets/yelp_bookstores_mock.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Load top-level businesses list with pd.DataFrame() (NOT pd.read_json())
bookstores_raw = pd.DataFrame(data["businesses"])
print("\nTop-level DataFrame (nested columns remain dicts/lists):")
print(bookstores_raw[["name", "categories", "coordinates"]].head(2))

Split-oriented JSON loaded into DataFrame:
   age_adjusted_death_rate  death_rate  deaths                                                     leading_cause              race_ethnicity sex  year
0                      7.6         6.2    32.0  Accidents Except Drug Poisoning (V01-X39, X43, X45-X59, Y85-Y86)  Asian and Pacific Islander   F  2007
1                      8.1         8.3    87.0  Accidents Except Drug Poisoning (V01-X39, X43, X45-X59, Y85-Y86)          Black Non-Hispanic   F  2007
2                      7.1         6.1    71.0  Accidents Except Drug Poisoning (V01-X39, X43, X45-X59, Y85-Y86)                    Hispanic   F  2007
3                      NaN         NaN     NaN  Accidents Except Drug Poisoning (V01-X39, X43, X45-X59, Y85-Y86)          Not Stated/Unknown   F  2007
4                      NaN         NaN     NaN  Accidents Except Drug Poisoning (V01-X39, X43, X45-X59, Y85-Y86)        Other Race/Ethnicity   F  2007

Top-level DataFrame (nested columns remain dicts/l

In [8]:
# 4.3 Flattening Nested & Deeply Nested JSON with pd.json_normalize()
# 1. Flatten single-level nested dicts (coordinates, location) with sep="_"
bookstores_flat = pd.json_normalize(data["businesses"], sep="_")
print("Flattened columns:", [c for c in bookstores_flat.columns if "coordinates" in c or "location" in c])

# 2. Flatten deeply nested list ("categories") and bring in parent metadata with meta & meta_prefix
categories_df = pd.json_normalize(
    data["businesses"],
    sep="_",
    record_path="categories",
    meta=[
        "name",
        "alias",
        "rating",
        ["coordinates", "latitude"],
        ["coordinates", "longitude"]
    ],
    meta_prefix="biz_"
)
print("\nDeeply normalized categories with biz_ metadata:")
print(categories_df)

Flattened columns: ['coordinates_latitude', 'coordinates_longitude', 'location_address1', 'location_address2', 'location_address3', 'location_city', 'location_zip_code', 'location_country', 'location_state']

Deeply normalized categories with biz_ metadata:
        alias               title                biz_name                             biz_alias biz_rating biz_coordinates_latitude biz_coordinates_longitude
0  bookstores          Bookstores   City Lights Bookstore   city-lights-bookstore-san-francisco        4.5                  37.7976               -122.406578
1  bookstores          Bookstores  Alexander Book Company  alexander-book-company-san-francisco        4.5                37.788585               -122.400631
2  stationery  Cards & Stationery  Alexander Book Company  alexander-book-company-san-francisco        4.5                37.788585               -122.400631
3  bookstores          Bookstores       Borderlands Books       borderlands-books-san-francisco        5.0    

In [9]:
# 4.4 Combining Multiple Datasets: pd.concat() and df.merge()
# 1. Concatenating paginated batches vertically with ignore_index=True
batch_1 = bookstores_flat.iloc[:2]
batch_2 = bookstores_flat.iloc[2:]
all_bookstores = pd.concat([batch_1, batch_2], ignore_index=True)
print("Concatenated bookstores shape:", all_bookstores.shape)

# 2. Merging call_counts with weather_daily on mismatched date column names
call_counts = pd.read_csv("datasets/call_counts.csv")
weather = pd.read_csv("datasets/weather_daily.csv")

merged = call_counts.merge(
    weather,
    left_on="created_date",
    right_on="date"
)
print("\nMerged 311 Call Counts with Daily Weather:")
print(merged)

Concatenated bookstores shape: (3, 17)

Merged 311 Call Counts with Daily Weather:
  created_date  call_counts        date  tmax  tmin
0   01/01/2018         4597  01/01/2018    19     7
1   01/02/2018         4362  01/02/2018    26    13
2   01/03/2018         3045  01/03/2018    30    16
3   01/04/2018         3374  01/04/2018    29    19
4   01/05/2018         4333  01/05/2018    19     9


### Chapter 4: Key Takeaways & Actionable Step-by-Step Process

#### Key Takeaways
* Specify `orient="split"` (or `"records"`, `"columns"`, `"index"`) in `pd.read_json()` whenever JSON files deviate from standard record orientation.
* Never pass a Python dictionary returned by `response.json()` into `pd.read_json()`; use `pd.DataFrame()` for flat lists of dicts or `pd.json_normalize()` for nested structures.
* Use `pd.json_normalize(..., record_path=..., meta=..., meta_prefix=...)` to unnest 1-to-many child arrays while preserving parent attributes without column name collisions.
* Stack paginated API batches with `pd.concat(..., ignore_index=True)` and enrich tables horizontally using `df.merge(other, left_on=..., right_on=...)`.

#### Step-by-Step Process to Act On
1. **Inspect JSON Structure**: Check whether raw JSON is record-oriented (`[...]`), split-oriented (`{"columns": ..., "data": ...}`), or deeply nested.
2. **Authenticate & Fetch via `requests`**: Pass query parameters via `params={...}` and API keys via `headers={"Authorization": "Bearer ..."}`, then parse with `data = response.json()`.
3. **Flatten Nested Payloads**: Call `pd.json_normalize(data["records"], sep="_", record_path=..., meta=..., meta_prefix=...)` to produce a tidy, flat `DataFrame`.
4. **Combine & Enrich**: Use `pd.concat(..., ignore_index=True)` across API pages and `df.merge(..., left_on=..., right_on=...)` to join external reference datasets.

---
### Chapter 4: Glossary & Official Documentation

| Item | Category | Explanation | Arguments Accepted | Official Documentation |
| :--- | :--- | :--- | :--- | :--- |
| `pd.read_json()` | `pandas` I/O Function | Converts a JSON string or file path into a `pandas` `DataFrame` or `Series`. | `path_or_buf`, `orient=None`, `dtype=None`, `convert_dates=True` | [Pandas `read_json` Docs](https://pandas.pydata.org/docs/reference/api/pandas.read_json.html) |
| `requests.get()` | `requests` Function | Sends an HTTP `GET` request to an API endpoint and returns a `Response` object. | `url`, `params=None`, `headers=None`, `timeout=None` | [Requests `get()` Docs](https://requests.readthedocs.io/en/latest/api/#requests.get) |
| `Response.json()` | `requests` Method | Decodes the JSON-encoded content of an HTTP response into a Python `dict` or `list`. | `**kwargs` (passed to `json.loads`) | [Requests `Response.json` Docs](https://requests.readthedocs.io/en/latest/api/#requests.Response.json) |
| `pd.json_normalize()` | `pandas` Function | Normalizes semi-structured JSON data into a flat tabular `DataFrame`. | `data`, `record_path=None`, `meta=None`, `meta_prefix=None`, `sep='.'` | [Pandas `json_normalize` Docs](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html) |
| `DataFrame.merge()` | `pandas` Method | Merges `DataFrame` objects using a database-style join on key columns or indices. | `right`, `how='inner'`, `on=None`, `left_on=None`, `right_on=None` | [Pandas `DataFrame.merge` Docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) |